In [9]:
import pandas as pd
import plotly.express as px
import os

# Load dataset
base_path = os.path.dirname(os.path.abspath(''))
features_path = os.path.join(base_path, 'data', 'features', 'features_ready.csv')
df = pd.read_csv(features_path)


C:\Users\aditi\AppData\Local\Temp\ipykernel_28832\2130060331.py:8: DtypeWarning:

Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.



In [10]:
# Add win flag if not already present
if 'win' not in df.columns:
    df['win'] = df['positionOrder'].apply(lambda x: 1 if x == 1 else 0)


In [11]:
#Top Constructor–Driver Pairings by Average Points
grouped = df.groupby(['constructor_name', 'driver_name'])['points'].mean().reset_index()
top_pairs = grouped.sort_values('points', ascending=False).head(20)

fig = px.bar(
    top_pairs,
    x='points',
    y='driver_name',
    color='constructor_name',
    orientation='h',
    title='Top 20 Constructor–Driver Pairings (Avg Points per Race)',
    labels={'points': 'Avg Points'},
    hover_data=['constructor_name']
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()


In [12]:
#Best Constructor–Driver Pairings by Total Points
pair_points = df.groupby(['constructor_name', 'driver_name'])['points'].sum().reset_index()
top_pairings = pair_points.sort_values('points', ascending=False).head(20)

fig = px.bar(
    top_pairings,
    x='points',
    y='driver_name',
    color='constructor_name',
    orientation='h',
    title='Top 20 Constructor–Driver Pairings by Total Points',
    labels={'points': 'Total Points'}
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()


In [13]:
#Constructor Win Rate Over Seasons
win_rate = df.groupby(['year', 'constructor_name'])['win'].mean().reset_index()

fig = px.line(
    win_rate,
    x='year',
    y='win',
    color='constructor_name',
    title='Constructor Win Rate per Season',
    labels={'win': 'Win Rate'}
)
fig.show()


In [14]:
#Average DNF Rate by Constructor
dnf_rate = df.groupby('constructor_name')['dnf_flag'].mean().reset_index().sort_values('dnf_flag', ascending=False)

fig = px.bar(
    dnf_rate,
    x='constructor_name',
    y='dnf_flag',
    title='Average DNF Rate by Constructor',
    labels={'dnf_flag': 'DNF Rate'}
)
fig.update_layout(xaxis={'categoryorder': 'total descending'})
fig.show()


In [15]:
#Average Constructor Points per Season

avg_team_pts = df.groupby(['year', 'constructor_name'])['points'].mean().reset_index()

fig = px.line(
    avg_team_pts,
    x='year',
    y='points',
    color='constructor_name',
    title='Average Constructor Points per Season',
    labels={'points': 'Avg Points'}
)
fig.show()


In [16]:
#K-Means Clustering of Constructors by Tier
from sklearn.cluster import KMeans

cluster_df = df.groupby('constructor_name')[['points', 'win']].mean().reset_index()
cluster_data = cluster_df[['points', 'win']]

kmeans = KMeans(n_clusters=3, random_state=42)
cluster_df['tier'] = kmeans.fit_predict(cluster_data)

fig = px.scatter(
    cluster_df,
    x='points',
    y='win',
    color='tier',
    hover_name='constructor_name',
    title='Constructor Performance Tiers (Clustered)',
    labels={'points': 'Avg Points', 'win': 'Win Rate'}
)
fig.show()
